In [1]:
%pip install langgraph langchain langchain-openAI tavily-python graphviz matplotlib python-dotenv ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
#create users, availability, appointments csvs
users_df = pd.DataFrame([
    {"user_id": "user_123", "patient_name": "Tendai Ball", "date_of_birth": "2003-01-01", "phone": "555-1234"},
    {"user_id": "user_456", "patient_name": "Jane Doe", "date_of_birth": "1998-07-10", "phone": "555-9876"},
    {"user_id": "user_789", "patient_name": "John Smith", "date_of_birth": "1985-11-22", "phone": "555-2468"}
])

availability_df = pd.DataFrame([
    {"date": "2026-04-15", "specialty": "cardiology",  "time": "9:00 AM",  "is_available": 1},
    {"date": "2026-04-15", "specialty": "cardiology",  "time": "1:30 PM",  "is_available": 1},
    {"date": "2026-04-15", "specialty": "dermatology", "time": "10:00 AM", "is_available": 1},
    {"date": "2026-04-15", "specialty": "dermatology", "time": "3:00 PM",  "is_available": 1},
    {"date": "2026-04-16", "specialty": "cardiology",  "time": "11:00 AM", "is_available": 1},
    {"date": "2026-04-16", "specialty": "neurology",   "time": "2:00 PM",  "is_available": 1}
])

appointments_df = pd.DataFrame([
    {
        "appointment_id": "APT-1001",
        "user_id": "user_123",
        "patient_name": "Tendai Ball",
        "date": "2026-04-15",
        "specialty": "cardiology",
        "time": "9:00 AM",
        "status": "booked"
    }
])

users_df.to_csv("users.csv", index=False)
availability_df.to_csv("availability.csv", index=False)
appointments_df.to_csv("appointments.csv", index=False)

print("Created users.csv, availability.csv, and appointments.csv")

Created users.csv, availability.csv, and appointments.csv


In [3]:
from dotenv import load_dotenv
import os

# Specify the full path to your .env file
load_dotenv(r"C:\Users\Tendai\OneDrive\Chat Agents\.env")
openai_api_key = os.getenv("OPENAI_API_KEY")
tavily_api_key = os.getenv("TAVILY_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = 'gpt-4o-mini', temperature=0, openai_api_key=openai_api_key)

In [5]:
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class BookingConfirmation:
    appointment_id: str = field(default="")
    user_id: str = field(default="")
    patient_name: str = field(default="")
    date: str = field(default="")
    specialty: str = field(default="")
    time: str = field(default="")
    status: Literal["booked", "pending", "cancelled"] = field(default="pending")

    def is_complete(self) -> bool:
        return all([self.appointment_id, self.user_id,
                    self.patient_name, self.date,
                    self.specialty, self.time])

    def missing_fields(self) -> list[str]:
        checks = {
            "appointment_id": self.appointment_id,
            "user_id": self.user_id,
            "patient_name": self.patient_name,
            "date": self.date,
            "specialty": self.specialty,
            "time": self.time,
        }
        return [k for k, v in checks.items() if not v]

In [36]:
import random
from langchain_core.tools import tool

def make_tools(booking: BookingConfirmation):

    @tool
    def get_user_id(patient_name: str) -> str:
        """Look up user ID from patient name."""
        db = pd.read_csv("users.csv")
        name_clean = patient_name.lower().strip()
        match = db[db["patient_name"].str.lower().str.strip().values == name_clean]
        if match.empty:
            return f"No user found with name: {patient_name}. Please ask the user to confirm their full name."
        row = match.iloc[0]
        booking.user_id = row["user_id"]
        booking.patient_name = row["patient_name"]
        return f"Found user_id: {row['user_id']} for {row['patient_name']}"

    @tool
    def generate_appointment_id() -> str:
        """Generate a unique appointment ID in format APPT_XXXX."""
        df = pd.read_csv("appointments.csv")
        existing = set(df["appointment_id"].tolist())
        while True:
            appt_id = f"APPT_{random.randint(0, 9999):04d}"
            if appt_id not in existing:
                booking.appointment_id = appt_id
                return f"Generated appointment_id: {appt_id}"


    @tool
    def check_availability(specialty: str, date: str) -> str:
        """Check available time slots for a specialty on a given date. Date must be in YYYY-MM-DD format."""
        cal = pd.read_csv("availability.csv")
        slots = cal[
            (cal["date"] == date) &
            (cal["specialty"] == specialty) &
            (cal["is_available"] == 1)       # only show open slots
      ]["time"].tolist()
        booking.specialty = specialty
        booking.date = date
        if not slots:
           return f"No available slots for {specialty} on {date}."
        return f"Available slots: {slots}"

    @tool
    def confirm_slot(time: str) -> str:
        """Confirm and lock a specific time slot for the appointment."""
        booking.time = time
        booking.status = "booked"
        return f"Slot confirmed at {time}"

    @tool
    def get_booking_status() -> str:
        """Check which fields are still missing from the booking."""
        missing = booking.missing_fields()
        if not missing:
            return "Booking is complete."
        return f"Still missing: {missing}"

    @tool
    def save_booking() -> str:
        """If and only if booking is complete, save it to appointments.csv."""
        if not booking.is_complete():
            return "Cannot save — missing: " + ", ".join(booking.missing_fields())
        new_entry = {
            "appointment_id": booking.appointment_id,
            "user_id": booking.user_id,
            "patient_name": booking.patient_name,
            "date": booking.date,
            "specialty": booking.specialty,
            "time": booking.time,
            "status": booking.status,
        }
        df = pd.read_csv("appointments.csv")
        df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
        df.to_csv("appointments.csv", index=False)
        return f"Booking saved: {booking.appointment_id}"

    return [get_user_id, generate_appointment_id, check_availability,
            confirm_slot, get_booking_status, save_booking]

In [7]:
import json
from langgraph.graph import MessagesState, START, END, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_MSG = SystemMessage(content=
    "You are a scheduling assistant. "
    "Extract the patient name from whatever phrasing the user provides (e.g. 'my name is X', 'I am X') "
    "and pass just the name to get_user_id. "
    "Use tools to fill all booking fields in this order: get_user_id, generate_appointment_id, "
    "check_availability, confirm_slot, save_booking. "
    "After each tool call use get_booking_status to check progress. "
    "Once get_booking_status returns 'Booking is complete', stop calling tools and confirm with the user. "
    "Never give medical advice. Ask follow-up questions only for info tools cannot provide."
)

def build_agent(tools):
    bound_llm = llm.bind_tools(tools)

    def llm_call(state: MessagesState):
        return {"messages": [bound_llm.invoke([SYSTEM_MSG] + state["messages"])]}

    def should_continue(state: MessagesState):
        return "action" if state["messages"][-1].tool_calls else END

    builder = StateGraph(MessagesState)
    builder.add_node("llm_call", llm_call)
    builder.add_node("environment", ToolNode(tools))
    builder.add_edge(START, "llm_call")
    builder.add_conditional_edges("llm_call", should_continue, {"action": "environment", END: END})
    builder.add_edge("environment", "llm_call")

    return builder.compile(checkpointer=InMemorySaver())

In [9]:
# Per-thread state — booking and agent are isolated per thread
_threads: dict[str, dict] = {}

def chat(thread_id: str, user_text: str):
    # Create fresh booking + agent for new threads
    if thread_id not in _threads:
        booking = BookingConfirmation()
        tools = make_tools(booking)
        _threads[thread_id] = {
            "booking": booking,
            "agent": build_agent(tools)
        }

    agent = _threads[thread_id]["agent"]
    config = {"configurable": {"thread_id": thread_id}}

    response = agent.invoke(
        {"messages": [HumanMessage(content=user_text)]},
        config=config
    )
    print(response["messages"][-1].content)

def reset_thread(thread_id: str):
    _threads.pop(thread_id, None)
    print(f"Thread '{thread_id}' reset.")

def get_booking(thread_id: str) -> BookingConfirmation | None:
    return _threads.get(thread_id, {}).get("booking")

In [35]:
def save_thread(thread_id: str, path: str | None = None):
    """Save a thread's message history to a JSON file and update availability."""
    if thread_id not in _threads:
        print(f"No thread found with id: {thread_id}")
        return

    agent = _threads[thread_id]["agent"]
    booking = _threads[thread_id]["booking"]
    config = {"configurable": {"thread_id": thread_id}}
    state = agent.get_state(config).values

    # --- Update availability.csv ---
    if booking.is_complete():
        availability = pd.read_csv("availability.csv")
        mask = (
            (availability["date"] == booking.date) &
            (availability["specialty"] == booking.specialty) &
            (availability["time"] == booking.time)
        )
        if mask.any():
            availability.loc[mask, "is_available"] = 0
            availability.to_csv("availability.csv", index=False)
            print(f"Marked {booking.specialty} slot on {booking.date} at {booking.time} as unavailable.")
        else:
            print("Warning: booked slot not found in availability.csv — no update made.")

    # --- Serialize and save JSON ---
    def serialize_message(msg):
        return {
            "type": msg.__class__.__name__,
            "content": str(msg.content),
        }

    payload = {
        "thread_id": thread_id,
        "booking": {
            "appointment_id": booking.appointment_id,
            "user_id": booking.user_id,
            "patient_name": booking.patient_name,
            "date": booking.date,
            "specialty": booking.specialty,
            "time": booking.time,
            "status": booking.status,
        },
        "messages": [serialize_message(m) for m in state["messages"]],
    }

    file_path = path or f"{thread_id}.json"
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, ensure_ascii=False)
    print(f"Thread saved to {file_path}")

In [46]:
chat("thread_06", "save my booking")

Your booking has been successfully saved. If you have any more questions or need further assistance, feel free to ask!


In [47]:
save_thread("thread_06")

Marked cardiology slot on 2026-04-16 at 11:00 AM as unavailable.
Thread saved to thread_06.json
